# CandyBench Notebook

可视化选择并并行测试 OpenAI-compatible 中转站模型。

所有实现均在本 Notebook 中。可直接修改下方配置、Prompt、答案解析和界面代码。

In [1]:
# ===== 可直接修改的配置 =====
from __future__ import annotations

import html
import json
import os
import re
import time
import urllib.error
import urllib.request
from concurrent.futures import ThreadPoolExecutor, as_completed
from datetime import datetime
from pathlib import Path

import ipywidgets as widgets
import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display


PROJECT_DIR = Path.cwd()


def load_dotenv(path: Path) -> None:
    if not path.exists():
        return
    for raw_line in path.read_text(encoding="utf-8").splitlines():
        line = raw_line.strip()
        if not line or line.startswith("#") or "=" not in line:
            continue
        key, value = line.split("=", 1)
        key = key.strip()
        value = value.strip().strip('"').strip("'")
        if key and key not in os.environ:
            os.environ[key] = value


load_dotenv(PROJECT_DIR / ".env")

BASE_URL = os.getenv("A6API_BASE_URL", "https://a6api.com/v1")
API_KEY = os.getenv("A6API_KEY") or os.getenv("OPENAI_API_KEY") or ""
CONCURRENCY = 10
TEMPERATURE = 0.0
EXPECTED_ANSWER = 21
OUTPUT_DIR = PROJECT_DIR / "outputs"

CANDY_PROMPT = """在一个不透明的黑袋子里装有三种口味的糖果：苹果味、桃子味和西瓜味。每种口味的糖果都有两种形状：圆形和五角星形。参赛者摸糖时不能分辨口味，但可以凭手感分辨形状，并据此选择摸取圆形或五角星形糖果。

袋中各类糖果数量如下：

圆形：苹果味 7 颗，桃子味 9 颗，西瓜味 8 颗；
五角星形：苹果味 7 颗，桃子味 6 颗，西瓜味 4 颗。

问：参赛者在活动前至少要决定摸出多少颗糖，才能保证手中至少有一对糖果，其中一颗是苹果味、另一颗是桃子味，且两颗糖果的形状不同？
也就是说，至少保证出现以下两种情况之一：

圆形苹果味与五角星形桃子味；
圆形桃子味与五角星形苹果味。"""

print(f"Base URL: {BASE_URL}")
print(f"API Key: {'已从 .env 加载' if API_KEY else '未配置，请在界面输入'}")
print(f"并发数: {CONCURRENCY} | 预期答案: {EXPECTED_ANSWER}")

Base URL: https://a6api.com/v1
API Key: 已从 .env 加载
并发数: 10 | 预期答案: 21


In [2]:
# ===== API、并发与判定逻辑，可直接修改 =====
class ApiError(RuntimeError):
    pass


def request_json(method, path, api_key, payload=None, timeout=120):
    url = f"{BASE_URL.rstrip('/')}{path}"
    body = None if payload is None else json.dumps(payload, ensure_ascii=False).encode("utf-8")
    request = urllib.request.Request(
        url,
        data=body,
        method=method,
        headers={
            "Authorization": f"Bearer {api_key}",
            "Content-Type": "application/json",
            "Accept": "application/json",
            "User-Agent": "CandyBenchNotebook/1.0",
        },
    )
    try:
        with urllib.request.urlopen(request, timeout=timeout) as response:
            raw = response.read().decode("utf-8")
    except urllib.error.HTTPError as exc:
        detail = exc.read().decode("utf-8", errors="replace")
        raise ApiError(f"HTTP {exc.code}: {detail}") from exc
    except urllib.error.URLError as exc:
        raise ApiError(f"Network error: {exc.reason}") from exc
    try:
        data = json.loads(raw)
    except json.JSONDecodeError as exc:
        raise ApiError(f"Response is not JSON: {raw[:500]}") from exc
    if not isinstance(data, dict):
        raise ApiError(f"Unexpected response: {type(data).__name__}")
    return data


def list_models(api_key):
    data = request_json("GET", "/models", api_key)
    models = []
    for item in data.get("data", []):
        if isinstance(item, dict) and isinstance(item.get("id"), str):
            models.append(item["id"])
        elif isinstance(item, str):
            models.append(item)
    return sorted(set(models))


def call_model(api_key, model, temperature):
    payload = {
        "model": model,
        "messages": [
            {"role": "system", "content": "你是严谨的数学推理助手。请给出清晰、简洁的中文推理。"},
            {"role": "user", "content": CANDY_PROMPT},
        ],
        "temperature": temperature,
    }
    data = request_json("POST", "/chat/completions", api_key, payload)
    choices = data.get("choices")
    if not isinstance(choices, list) or not choices:
        raise ApiError(f"No choices: {json.dumps(data, ensure_ascii=False)[:500]}")
    first = choices[0] if isinstance(choices[0], dict) else {}
    message = first.get("message", {})
    content = message.get("content") if isinstance(message, dict) else None
    content = content if isinstance(content, str) else first.get("text")
    if not isinstance(content, str):
        raise ApiError(f"No text content: {json.dumps(data, ensure_ascii=False)[:500]}")
    return content.strip()


def call_and_capture(api_key, model, temperature):
    started = time.time()
    try:
        return model, "ok", call_model(api_key, model, temperature), time.time() - started
    except Exception as exc:
        return model, "error", str(exc), time.time() - started


def extract_final_answer(content):
    tail = content[-1600:]
    patterns = (
        r"(?:最终答案|最终结果|答案|至少(?:需要|要)?|合计|总计)\s*(?:是|为|：|:)?\s*[^\d]{0,12}(\d+)\s*(?:颗|个)?",
        r"(?:final answer|answer|minimum|total)\s*(?:is|:|=)?\s*[^\d]{0,12}(\d+)",
        r"\\boxed\{\s*(\d+)\s*\}",
    )
    candidates = []
    for pattern in patterns:
        for match in re.finditer(pattern, tail, flags=re.IGNORECASE):
            candidates.append((match.start(), int(match.group(1))))
    if candidates:
        return max(candidates, key=lambda item: item[0])[1]
    final_lines = re.findall(r"(?m)^\s*(\d+)\s*(?:颗|个)?[。.!！]?\s*$", tail)
    return int(final_lines[-1]) if final_lines else None


def append_result(path, result):
    model, status, content, elapsed = result
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("a", encoding="utf-8") as file:
        file.write("=" * 88 + "\n")
        file.write(f"Time: {datetime.now().isoformat(timespec='seconds')}\n")
        file.write(f"Model: {model}\nStatus: {status}\nElapsed seconds: {elapsed:.2f}\n")
        file.write("-" * 88 + "\n" + content.strip() + "\n\n")


def classify_result(result):
    model, status, content, elapsed = result
    answer = extract_final_answer(content) if status == "ok" else None
    if status != "ok":
        verdict = "ERROR"
    elif answer is None:
        verdict = "UNKNOWN"
    elif answer == EXPECTED_ANSWER:
        verdict = "PASS"
    else:
        verdict = "FAIL"
    return {
        "模型": model,
        "状态": status,
        "解析答案": answer,
        "判定": verdict,
        "耗时(秒)": round(elapsed, 2),
    }

In [3]:
# ===== 可视化界面，可直接修改 =====
model_checks = {}
results = []

base_url_input = widgets.Text(value=BASE_URL, description="Base URL", layout=widgets.Layout(width="460px"))
key_input = widgets.Password(
    placeholder="已从 .env 加载" if API_KEY else "输入 API Key",
    description="API Key",
    layout=widgets.Layout(width="360px"),
)
search_input = widgets.Text(placeholder="筛选模型", description="搜索", layout=widgets.Layout(width="360px"))
concurrency_input = widgets.IntSlider(value=CONCURRENCY, min=1, max=30, description="并发")
temperature_input = widgets.FloatSlider(value=TEMPERATURE, min=0, max=1, step=0.1, description="温度")

load_button = widgets.Button(description="加载模型", icon="refresh", button_style="info")
select_button = widgets.Button(description="全选", icon="check-square-o")
clear_button = widgets.Button(description="清空", icon="square-o")
run_button = widgets.Button(description="运行测试", icon="play", button_style="success", disabled=True)
selection_count = widgets.HTML("<b>已选 0</b>")
status_view = widgets.HTML("等待加载模型")
summary_view = widgets.HTML()
progress_view = widgets.IntProgress(value=0, min=0, max=1, description="进度")

model_grid = widgets.GridBox(
    [],
    layout=widgets.Layout(
        grid_template_columns="repeat(auto-fit, minmax(240px, 1fr))",
        grid_gap="4px 12px",
        max_height="420px",
        overflow="auto",
        border="1px solid #d0d7de",
        padding="8px",
    ),
)
table_output = widgets.Output()
chart_output = widgets.Output()
reply_output = widgets.Output()
result_tabs = widgets.Tab(children=[table_output, chart_output, reply_output])
for index, title in enumerate(("结果表", "耗时图", "完整回复")):
    result_tabs.set_title(index, title)

prompt_panel = widgets.Accordion(
    children=[widgets.HTML(f"<pre style='white-space:pre-wrap;margin:0'>{html.escape(CANDY_PROMPT)}</pre>")],
    selected_index=None,
)
prompt_panel.set_title(0, "固定测试题")


def current_key():
    return key_input.value.strip() or API_KEY


def selected_models():
    return [model for model, checkbox in model_checks.items() if checkbox.value]


def visible_models():
    query = search_input.value.strip().lower()
    return [model for model in model_checks if not query or query in model.lower()]


def update_selection(*_):
    count = len(selected_models())
    selection_count.value = f"<b>已选 {count}</b>"
    run_button.disabled = count == 0


def apply_filter(*_):
    visible = set(visible_models())
    for model, checkbox in model_checks.items():
        checkbox.layout.display = "" if model in visible else "none"


def load_models(*_):
    global BASE_URL, model_checks
    api_key = current_key()
    if not api_key:
        status_view.value = "<span style='color:#b3261e'>缺少 API Key</span>"
        return
    BASE_URL = base_url_input.value.strip().rstrip("/")
    load_button.disabled = True
    status_view.value = "正在加载模型..."
    try:
        models = list_models(api_key)
        model_checks = {}
        children = []
        for model in models:
            checkbox = widgets.Checkbox(value=False, description=model, indent=False)
            checkbox.observe(update_selection, names="value")
            model_checks[model] = checkbox
            children.append(checkbox)
        model_grid.children = tuple(children)
        status_view.value = f"已加载 <b>{len(models)}</b> 个模型"
        apply_filter()
        update_selection()
    except Exception as exc:
        status_view.value = f"<span style='color:#b3261e'>加载失败：{html.escape(str(exc))}</span>"
    finally:
        load_button.disabled = False


def select_visible(*_):
    for model in visible_models():
        model_checks[model].value = True
    update_selection()


def clear_selection(*_):
    for checkbox in model_checks.values():
        checkbox.value = False
    update_selection()


def render_results(report_path):
    frame = pd.DataFrame([classify_result(result) for result in results])
    frame = frame.sort_values(["判定", "模型"], kind="stable")
    counts = frame["判定"].value_counts()
    summary_view.value = (
        f"<b>PASS {counts.get('PASS', 0)}</b> · FAIL {counts.get('FAIL', 0)} · "
        f"UNKNOWN {counts.get('UNKNOWN', 0)} · ERROR {counts.get('ERROR', 0)}"
    )

    table_output.clear_output()
    with table_output:
        display(frame.reset_index(drop=True))

    chart_frame = frame.sort_values("耗时(秒)", ascending=False).head(25).sort_values("耗时(秒)")
    colors = {"PASS": "#137333", "FAIL": "#b3261e", "UNKNOWN": "#8a6d1d", "ERROR": "#5f6368"}
    chart_output.clear_output()
    with chart_output:
        figure, axis = plt.subplots(figsize=(10, max(4, 0.34 * len(chart_frame))))
        axis.barh(chart_frame["模型"], chart_frame["耗时(秒)"], color=[colors[x] for x in chart_frame["判定"]])
        axis.set_title("模型响应耗时（最慢 25 个）" if len(frame) > 25 else "模型响应耗时")
        axis.set_xlabel("秒")
        axis.grid(axis="x", alpha=0.2)
        figure.tight_layout()
        plt.show()

    children, titles = [], []
    for model, status, content, elapsed in sorted(results, key=lambda item: item[0].lower()):
        answer = extract_final_answer(content) if status == "ok" else None
        children.append(widgets.HTML(f"<pre style='white-space:pre-wrap;margin:0'>{html.escape(content)}</pre>"))
        titles.append(f"{model} · {status} · answer {answer if answer is not None else '?'} · {elapsed:.2f}s")
    accordion = widgets.Accordion(children=children, selected_index=None)
    for index, title in enumerate(titles):
        accordion.set_title(index, title)
    reply_output.clear_output()
    with reply_output:
        display(accordion)

    status_view.value = f"完成 <b>{len(results)}</b> 个模型 · TXT：<code>{html.escape(str(report_path))}</code>"


def run_selected(*_):
    global results
    models = selected_models()
    api_key = current_key()
    if not models or not api_key:
        return
    report_path = OUTPUT_DIR / f"candy_model_replies_{datetime.now():%Y%m%d_%H%M%S}.txt"
    results = []
    progress_view.max = len(models)
    progress_view.value = 0
    run_button.disabled = True
    status_view.value = f"正在测试 <b>{len(models)}</b> 个模型..."
    try:
        workers = max(1, min(concurrency_input.value, len(models)))
        with ThreadPoolExecutor(max_workers=workers) as executor:
            futures = {
                executor.submit(call_and_capture, api_key, model, temperature_input.value): model
                for model in models
            }
            for completed, future in enumerate(as_completed(futures), start=1):
                result = future.result()
                results.append(result)
                append_result(report_path, result)
                model, state, _, elapsed = result
                progress_view.value = completed
                status_view.value = f"[{completed}/{len(models)}] <code>{html.escape(model)}</code> · {state} · {elapsed:.2f}s"
        render_results(report_path)
    finally:
        run_button.disabled = False


load_button.on_click(load_models)
select_button.on_click(select_visible)
clear_button.on_click(clear_selection)
run_button.on_click(run_selected)
search_input.observe(apply_filter, names="value")

app = widgets.VBox(
    [
        widgets.HTML("<h2 style='margin:0'>CandyBench Notebook</h2><div style='color:#57606a'>糖果推理测试 · 预期答案 21</div>"),
        widgets.HBox([base_url_input, key_input, load_button]),
        widgets.HBox([search_input, select_button, clear_button, selection_count]),
        model_grid,
        widgets.HBox([concurrency_input, temperature_input, run_button]),
        progress_view,
        status_view,
        summary_view,
        result_tabs,
        prompt_panel,
    ],
    layout=widgets.Layout(gap="10px", width="100%"),
)

display(app)
if API_KEY:
    load_models()